In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import dotenv
import dspy

In [ ]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.pardir, os.path.pardir, os.path.pardir))
DOTENV_FILE = os.path.join(PROJECT_ROOT, '.env')


dotenv.load_dotenv(DOTENV_FILE)

# Language Models
The first step in any DSPy code is to set up your language model. For example, you can configure OpenAI's GPT-4o-mini as your default LM as follows.

In [ ]:
lm = dspy.LM('openai/gpt-4o-mini')
dspy.configure(lm=lm)

## Calling the LM directly.
It's easy to call the lm you configured above directly. This gives you a unified API and lets you benefit from utilities like automatic caching.

In [ ]:
# Calling the LM directly

response = lm("Say this is a test!", temperature=0.7)  # => ['This is a test!']
print(response)

chat_response = lm(messages=[{"role": "user", "content": "Say this is a test!"}])  # => ['This is a test!']
print(chat_response)

## Using the LM with DSPy modules.
Idiomatic DSPy involves using modules, which we discuss in the next guide.

In [ ]:
# Define a module (ChainOfThought) and assign it a signature (return an answer, given a question).
qa = dspy.ChainOfThought('question -> answer')

# Run with the default LM configured with `dspy.configure` above.
response = qa(question="How many floors are in the castle David Gregory inherited?")
print(response.answer)

# Using multiple LMs.
You can change the default LM globally with dspy.configure or change it inside a block of code with dspy.context.

!Note: Using dspy.configure and dspy.context is thread-safe!

In [ ]:
dspy.configure(lm=dspy.LM('openai/gpt-4o-mini'))
response = qa(question="How many floors are in the castle David Gregory inherited?")
print('GPT-4o-mini:', response.answer)

with dspy.context(lm=dspy.LM('openai/gpt-3.5-turbo')):
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print('GPT-3.5-turbo:', response.answer)

## Configuring LM generation.
For any LM, you can configure any of the following attributes at initialization or in each subsequent call.

In [ ]:
gpt_4o_mini = dspy.LM('openai/gpt-4o-mini', temperature=0.9, max_tokens=3000, stop=None, cache=False)

By default LMs in DSPy are cached. If you repeat the same call, you will get the same outputs. But you can turn off caching by setting cache=False.

If you want to keep caching enabled but force a new request (for example, to obtain diverse outputs), pass a unique rollout_id and set a non-zero temperature in your call. DSPy hashes both the inputs and the rollout_id when looking up a cache entry, so different values force a new LM request while still caching future calls with the same inputs and rollout_id. The ID is also recorded in lm.history, which makes it easy to track or compare different rollouts during experiments. Changing only the rollout_id while keeping temperature=0 will not affect the LM's output.

In [ ]:
lm("Say this is a test!", rollout_id=1, temperature=1.0)

You can pass these LM kwargs directly to DSPy modules as well. Supplying them at initialization sets the defaults for every call:

In [ ]:
predict = dspy.Predict("question -> answer", rollout_id=1, temperature=1.0)

To override them for a single invocation, provide a config dictionary when calling the module:

In [ ]:
predict = dspy.Predict("question -> answer")
predict(question="What is 1 + 52?", config={"rollout_id": 5, "temperature": 1.0})

In both cases, rollout_id is forwarded to the underlying LM, affects its caching behavior, and is stored alongside each response so you can replay or analyze specific rollouts later.

## Inspecting output and usage metadata.
Every LM object maintains the history of its interactions, including inputs, outputs, token usage (and $$$ cost), and metadata.

In [ ]:
print(len(lm.history))  # e.g., 3 calls to the LM

lm.history[-1].keys()  # access the last call to the LM, with all metadata

## Using the Responses API
By default, DSPy calls language models (LMs) using LiteLLM's [Chat Completions API](https://docs.litellm.ai/docs/completion), which is suitable for most standard models and tasks. However, some advanced models, such as OpenAI's reasoning models (e.g., gpt-5 or other future models), may offer improved quality or additional features when accessed via the [Responses API](https://docs.litellm.ai/docs/response_api), which is supported in DSPy.

**When should you use the Responses API?**

- If you are working with models that support or require the responses endpoint (such as OpenAI's reasoning models).
- When you want to leverage enhanced reasoning, multi-turn, or richer output capabilities provided by certain models.

**How to enable the Responses API in DSPy:**

To enable the Responses API, just set model_type="responses" when creating the dspy.LM instance.

In [ ]:
# Configure DSPy to use the Responses API for your language model
dspy.settings.configure(
    lm=dspy.LM(
        "openai/gpt-5-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=16000,
    ),
)

Please note that not all models or providers support the Responses API, check [LiteLLM's documentation](https://docs.litellm.ai/docs/response_api) for more details.

## Advanced: Building custom LMs and writing your own Adapters.
Though rarely needed, you can write custom LMs by inheriting from dspy.BaseLM. Another advanced layer in the DSPy ecosystem is that of adapters, which sit between DSPy signatures and LMs. A future version of this guide will discuss these advanced features, though you likely don't need them.